In [66]:
import pandas as pd
import numpy as np
import networkx as nx
import pickle
from typing import Dict, List, Optional, Set, Tuple
import re

from rule_engine import extract_rules, summarise_rules, KGRuleEngine

In [14]:
def load_graph(filepath):
    with open(filepath,'rb') as f:
        G = pickle.load(f)
    return G

In [36]:
CAUSAL_SIGN = {
    "increases":            +1,
    "directly_increases":   +1,
    "positive_correlation": +1,
    "decreases":            -1,
    "directly_decreases":   -1,
    "negative_correlation": -1,
}
AD_PATHOLOGY_NODES = {
    'path(MESH:"Alzheimer Disease")',
    'path(EFO:"late-onset Alzheimers disease")',
}
# BPs whose names contain these keywords are AD-relevant
AD_BP_KEYWORDS = {
    "amyloid", "tau", "neurofibrill", "alzheimer", "neuroinflam",
    "synap", "neuron death", "apoptosis", "oxidative stress",
    "mitochondri", "lipid peroxid", "memory", "cognitive",
    "microglial", "neurodegenerat", "autophagy", "ubiquitin",
}
NAME_RE = re.compile(r'"([^"]+)"')

def _node_name(node) -> str:
    m = NAME_RE.search(str(node))
    return m.group(1) if m else str(node)


def _edge_sign(rel: str) -> Optional[int]:
    rel = rel.lower()
    if rel.startswith("rev_"):
        rel = rel[4:]
    return CAUSAL_SIGN.get(rel)


def _is_gene_node(node) -> bool:
    s = str(node)
    return s.startswith(("g(", "p(", "r("))


def _is_bp_node(node) -> bool:
    return str(node).startswith("bp(")


def _is_ad_bp(node) -> bool:
    name = _node_name(node).lower()
    return any(kw in name for kw in AD_BP_KEYWORDS)


def _build_directed_graph(G) -> nx.DiGraph:
    DG = nx.DiGraph()
    for src, tgt, rel, _ in G.edges(keys=True, data=True):
        sign = _edge_sign(rel)
        if sign is None:
            continue
        if rel.lower().startswith("rev_"):
            src, tgt = tgt, src
        DG.add_edge(src, tgt, sign=sign, rel=rel)
    return DG

In [160]:
kg_path = "../data/KG/ad_network_with_reverse_edges.pkl"
G_ad = load_graph(kg_path)
DG_ad = _build_directed_graph(G_ad)

exp_path = '../data/ANMerge/ANMerge_TX_preprocessed/AnmergeTX_cleaned_exp.csv'
exp_df = pd.read_csv(exp_path, index_col=0)
expr_genes = set(exp_df.columns)

rules = extract_rules(kg_path = kg_path, expression_genes = expr_genes)
df_rules = summarise_rules(rules)
df_rules

  Direct gene→AD rules:    185
  Pathway activation rules: 9
  Causal chain rules:      547


,name,type,conclusion,n_genes,genes,bp,confidence,evidence
0,chain_SORL1_APP_CTL,direct_gene_ad,CTL,1,SORL1,,0.80,"g(HGNC:""SORL1"") → APP → Alzheimer Disease (net..."
1,chain_MME_APP_CTL,direct_gene_ad,CTL,1,MME,,0.80,"p(HGNC:""MME"") → APP → Alzheimer Disease (net s..."
2,chain_CTSD_APP_AD,direct_gene_ad,AD,1,CTSD,,0.80,"g(HGNC:""CTSD"",var(""C,224,T"")) → APP → Alzheime..."
3,direct_BIN1_AD,direct_gene_ad,AD,1,BIN1,,1.00,"g(HGNC:""BIN1"") --rev_positive_correlation--> p..."
4,direct_PICALM_AD,direct_gene_ad,AD,1,PICALM,,1.00,"g(HGNC:""PICALM"") --rev_positive_correlation-->..."
...,...,...,...,...,...,...,...,...
736,chain_AKAP13_4hops_AD,causal_chain,AD,1,AKAP13,,0.60,AKAP13 → MAPT → APP → Alzheimer Disease
737,chain_VPS35_5hops_AD,causal_chain,AD,1,VPS35,,0.45,VPS35 → APP → APP → TNF → Alzheimer Disease
738,chain_VPS35_5hops_CTL,causal_chain,CTL,1,VPS35,,0.45,VPS35 → APP → APP → nitric oxide → Alzheimer D...
739,chain_VPS35_4hops_AD,causal_chain,AD,1,VPS35,,0.60,VPS35 → APP → APP → Alzheimer Disease


## Evaluate rules

In [163]:
print("\nComputing eCDF scores...")
ecdf_path = "../EdgeAssignmentTask/neighborhood_selection/results/aggregator2/ANMerge/AD_KG/ecdf_5/sample_scoring_ecdf.csv"
ecdf = pd.read_csv(ecdf_path, index_col=0)
ctl_ids = list(ecdf[ecdf['label']==0].index)

exp = exp_df.T
ecdf_df = compute_ecdf_scores(exp, ctl_ids)
ecdf_df


Computing eCDF scores...


Patients,DCR00001_1,DCR00004_1,DCR00009_1,DCR00013_1,DCR00014_1,DCR00015_1,DCR00018_1,DCR00020_1,DCR00021_1,DCR00025_1,...,TLSMCI602_1,TLSMCI603_1,TLSMCI604_1,TLSMCI605_1,TLSMCI606_1,TLSMCI607_1,TLSMCI608_1,TLSMCI609_1,TLSMCI611_1,TLSMCI612_1
AAK1,-0.022088,-0.335341,-0.255020,0.066265,-0.018072,-0.371486,-0.463855,0.106426,-0.174699,0.030120,...,0.495984,0.495984,0.182731,0.475904,0.439759,0.475904,0.206827,-0.090361,0.090361,0.295181
AARS,-0.411647,-0.411647,-0.299197,-0.118474,-0.090361,-0.246988,-0.371486,-0.070281,-0.251004,-0.251004,...,0.479920,0.479920,0.335341,0.491968,0.459839,0.500000,0.106426,-0.110442,-0.050201,0.407631
AATF,-0.122490,-0.383534,-0.170683,-0.074297,-0.026104,-0.319277,-0.403614,-0.275100,-0.186747,0.014056,...,0.335341,0.491968,0.411647,0.471888,0.463855,0.439759,0.463855,-0.459839,0.022088,0.371486
ABCA1,-0.234940,-0.355422,-0.238956,-0.435743,-0.130522,-0.058233,-0.283133,-0.315261,-0.122490,-0.475904,...,0.291165,0.246988,0.455823,0.251004,0.238956,0.030120,0.315261,0.174699,-0.459839,0.335341
ABCA7,0.018072,-0.186747,-0.082329,-0.062249,0.062249,-0.166667,-0.347390,-0.182731,-0.158635,0.106426,...,0.395582,0.495984,0.495984,0.194779,0.367470,0.082329,0.383534,-0.363454,-0.130522,0.271084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZSWIM1,-0.242972,-0.367470,-0.259036,-0.130522,-0.074297,-0.222892,-0.138554,-0.146586,-0.114458,-0.287149,...,0.491968,0.495984,0.415663,0.455823,0.295181,0.226908,0.395582,-0.371486,-0.271084,0.495984
ZSWIM6,-0.423695,-0.050201,0.022088,-0.194779,-0.471888,-0.122490,-0.090361,-0.423695,-0.323293,-0.483936,...,0.070281,-0.022088,0.170683,0.070281,0.146586,0.070281,0.178715,-0.162651,-0.363454,0.106426
ZYG11B,-0.082329,-0.238956,-0.190763,-0.142570,0.010040,-0.042169,-0.363454,-0.206827,-0.138554,-0.150602,...,0.423695,0.479920,0.343373,0.399598,0.383534,0.283133,0.479920,-0.415663,-0.118474,0.455823
ZYX,-0.154618,-0.166667,-0.154618,-0.126506,-0.238956,-0.259036,-0.307229,-0.359438,-0.251004,-0.222892,...,0.467871,0.475904,0.427711,0.459839,0.226908,0.078313,0.495984,-0.335341,-0.070281,0.475904


In [97]:
print("Fitting rule engine...")
engine = KGRuleEngine(rules, threshold_std=1.0)
engine.fit(ecdf_df, ctl_ids)

print("Evaluating rules on all patients...")
results = engine.predict(ecdf_df, min_fired_rules=1)
results

Fitting rule engine...
Evaluating rules on all patients...


,patient_id,rules_fired,ad_rules,ctl_rules,ad_rate,ctl_rate,conclusion,top_ad_rules,top_ctl_rules
0,DCR00001_1,37,19,18,0.00893,0.01345,CTL,chain_SNCA_MAPT_AD; chain_IL10_IL6_AD; chain_I...,chain_MMP9_APP_CTL; chain_IL10_inflammatory re...
1,DCR00004_1,34,18,16,0.00899,0.01457,CTL,chain_SNCA_MAPT_AD; chain_IL10_IL6_AD; chain_I...,chain_IL10_inflammatory respons_CTL; chain_FOX...
2,DCR00009_1,59,41,18,0.02478,0.01509,AD,chain_HDAC2_memory_AD; direct_PTGS1_AD; chain_...,chain_HDAC2_H3F3A_CTL; chain_IL10_inflammatory...
3,DCR00013_1,26,17,9,0.00679,0.00457,AD,chain_MARCKS_APP_AD; chain_IL23A_inflammatory ...,direct_TNFRSF1B_CTL; chain_MARCKS_5hops_CTL; c...
4,DCR00014_1,17,8,9,0.00510,0.01101,CTL,pw_suppress_mitochondria-nucleus signaling pat...,chain_FOXO4_insulin receptor sig_CTL; direct_T...
...,...,...,...,...,...,...,...,...,...
686,TLSMCI607_1,423,229,194,0.15592,0.17723,CTL,direct_BIN1_AD; chain_CYLD_Wnt signaling pathw...,chain_SORL1_APP_CTL; chain_RTN4_APP_CTL; chain...
687,TLSMCI608_1,531,299,232,0.25206,0.28756,CTL,chain_CTSD_APP_AD; direct_PICALM_AD; direct_CD...,chain_SORL1_APP_CTL; chain_MME_APP_CTL; chain_...
688,TLSMCI609_1,39,22,17,0.00941,0.01349,CTL,chain_SNCA_MAPT_AD; chain_IL23A_inflammatory r...,chain_FOXO4_insulin receptor sig_CTL; direct_T...
689,TLSMCI611_1,36,21,15,0.01188,0.01462,CTL,chain_SNCA_MAPT_AD; chain_IL23A_inflammatory r...,chain_IL10_inflammatory respons_CTL; chain_FOX...


In [98]:
print("\nRule-based prediction summary:")
print(results["conclusion"].value_counts().to_string())

ecdf['label'] = ecdf['label'].map({1:'AD',0:'CTL'})
true_labels = ecdf['label']
results["true"] = true_labels.values

correct = (results["conclusion"] == results["true"]) & (results["conclusion"] != "uncertain")
covered = results["conclusion"] != "uncertain"
print(f"\nCoverage (patients with ≥1 rule fired): {covered.sum()} / {len(results)}")
if covered.sum() > 0:
    acc = correct.sum() / covered.sum()
    print(f"Accuracy on covered patients: {acc:.3f}")


Rule-based prediction summary:
conclusion
CTL    448
AD     243

Coverage (patients with ≥1 rule fired): 691 / 691
Accuracy on covered patients: 0.454


## ecdf score
+ empirical Culmulative Distribution:
+ empirical Culmulative Distribution Function:
+ this implementation:
+ Vinay implementation:

In [101]:
def compute_ecdf_scores(
    expression_df: pd.DataFrame,
    ctrl_columns: List[str],
) -> pd.DataFrame:
    """Per-gene eCDF scores for all patients using the control distribution.

    For each gene g and patient p:
        ecdf_score[g, p] = F_ctrl(x_{g,p}) - 0.5

    where F_ctrl is the empirical CDF fitted on ctrl_columns patients.
    Score is in [-0.5, 0.5]: 0 = at control median, +0.5 = above all controls,
    -0.5 = below all controls.

    Uses searchsorted for O(n_genes * n_ctrl * log(n_ctrl)) complexity.
    """
    expr = expression_df.copy()
    expr = expr[~expr.index.duplicated(keep='first')]

    ctrl_cols = [c for c in ctrl_columns if c in expr.columns]
    ctrl_expr = expr[ctrl_cols].values  # (n_genes, n_ctrl)
    all_expr = expr.values              # (n_genes, n_patients)
    n_genes, n_ctrl = ctrl_expr.shape
    n_patients = all_expr.shape[1]

    ecdf_matrix = np.empty((n_genes, n_patients), dtype=float)
    for g in range(n_genes):
        ctrl_sorted = np.sort(ctrl_expr[g])
        ecdf_matrix[g] = np.searchsorted(ctrl_sorted, all_expr[g], side='right') / n_ctrl

    score_matrix = ecdf_matrix - 0.5
    return pd.DataFrame(score_matrix, index=expr.index, columns=expr.columns)

exp = exp_df.T
ecdf_df = compute_ecdf_scores(exp, ctl_ids)
ecdf_df

,DCR00001_1,DCR00004_1,DCR00009_1,DCR00013_1,DCR00014_1,DCR00015_1,DCR00018_1,DCR00020_1,DCR00021_1,DCR00025_1,...,TLSMCI602_1,TLSMCI603_1,TLSMCI604_1,TLSMCI605_1,TLSMCI606_1,TLSMCI607_1,TLSMCI608_1,TLSMCI609_1,TLSMCI611_1,TLSMCI612_1
GIMAP4,-0.230924,0.026104,-0.126506,-0.194779,-0.114458,-0.098394,0.014056,-0.234940,-0.371486,-0.443775,...,0.050201,0.050201,0.154618,0.291165,0.146586,0.303213,0.094378,0.014056,-0.222892,0.098394
C1QBP,-0.423695,-0.355422,-0.307229,-0.403614,-0.279116,-0.226908,-0.182731,-0.246988,0.351406,0.471888,...,-0.058233,-0.118474,-0.014056,-0.086345,0.158635,0.255020,-0.138554,-0.287149,-0.467871,-0.034137
OCIAD2,-0.500000,-0.467871,-0.403614,-0.447791,-0.275100,-0.355422,-0.371486,-0.202811,-0.230924,-0.411647,...,0.074297,0.114458,0.086345,0.134538,0.122490,0.455823,0.046185,-0.062249,-0.479920,0.046185
ERICH1,-0.106426,-0.002008,-0.090361,-0.255020,-0.182731,-0.162651,0.014056,-0.094378,-0.255020,-0.150602,...,0.226908,0.046185,0.198795,0.122490,0.367470,0.182731,0.174699,0.006024,-0.090361,0.439759
MRTFA,-0.299197,-0.371486,-0.327309,-0.182731,-0.251004,-0.323293,-0.423695,-0.323293,-0.002008,0.070281,...,0.403614,0.471888,0.419679,0.419679,0.359438,0.142570,0.463855,-0.471888,-0.206827,0.471888
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SLC7A7,-0.078313,0.014056,0.022088,-0.082329,-0.259036,-0.126506,-0.062249,-0.042169,-0.435743,-0.455823,...,0.435743,0.295181,0.375502,0.435743,0.467871,0.122490,0.403614,-0.279116,-0.138554,0.303213
NAA10,-0.315261,-0.463855,-0.331325,-0.487952,-0.303213,-0.391566,-0.255020,-0.110442,0.443775,0.295181,...,0.251004,0.130522,0.251004,0.010040,0.066265,0.010040,0.038153,-0.339357,-0.198795,0.038153
ORAI1,-0.238956,-0.319277,-0.339357,-0.178715,-0.214859,-0.399598,-0.467871,-0.399598,0.126506,-0.162651,...,0.483936,0.467871,0.098394,0.487952,0.090361,0.435743,0.134538,-0.403614,-0.307229,0.218876
ZZeqdeed69UrlI7lNY,-0.259036,-0.411647,-0.038153,-0.423695,-0.343373,-0.311245,-0.427711,-0.363454,0.391566,0.283133,...,0.150602,0.246988,0.058233,0.210843,0.062249,0.086345,0.126506,-0.166667,-0.198795,0.226908


In [161]:
def compute_log2fc(expression_df, ctl_ids):
    """Without statistical testing of whether disease_group and control_group gene expression
       significantly or not. Simply calculate a continous logfc score for each patient's each gene
       expression: 
       logfc[gene,patient] = log2fc_gene_expression[gene,patient] - log2fc_gene_expression[gene,control_median]

    Args:
        expression_df (pd.DataFrame): [gene, sample_ids]
        ctl_ids (list): _description_

    Returns:
        np.array: _description_
    """
    # 1. Filter valid control columns and check if any exist
    ctl_cols   = [c for c in ctl_ids if c in expression_df.columns]
    if not ctl_cols:
        raise ValueError("None of the specified ctl_ids match the columns in expression_df.")
    ctl_median = expression_df[ctl_cols].median(axis=1)

    # 2. Safety check: Handle negative values using Min-Max scaling
    min_val = expression_df.min().min()
    if min_val < -1:
        # Min-Max scaling formula: (x - min) / (max - min), completely preventing log(<=0) errors.
        df_min = expression_df.min().min()
        df_max = expression_df.max().min() if expression_df.max().max() == expression_df.max().min() else expression_df.max().max()
        
        if df_max - df_min > 0:
            expression_df = (expression_df - df_min) / (df_max - df_min)
        else:
            expression_df = expression_df - df_min  # Fallback if flat data

    # 3. Compute control median safely (ignoring NaNs)
    ctl_median = expression_df[ctl_cols].median(axis=1, skipna=True)

    # 4. Compute log2 values safely
    log_expr = np.log2(expression_df.clip(lower=0) + 1)
    log_ctl_median = np.log2(ctl_median.clip(lower=0) + 1)

    # 5. Subtract and handle any remaining NaNs
    logfc_df = log_expr.subtract(log_ctl_median, axis=0)
    #logfc_df = logfc_df.fillna(0.0)

    return logfc_df

logfc_df = compute_log2fc(exp, ctl_ids)


In [164]:
def compute_kg_rule_scores(log2fc_df, rules):
    """Compute continuous KG rule activation score per patient per rule.

    Score = mean(sign_g * log2fc[g]) for genes in the rule.
    Positive score means the rule's causal direction is supported.
    """
    patients = log2fc_df.columns.tolist()
    scores   = {}

    for rule in rules:
        signals = []
        for gene, sign in zip(rule.genes, rule.signs):
            g = gene.upper()
            if g in log2fc_df.index:
                signals.append(log2fc_df.loc[g] * sign)
        if not signals:
            continue
        avg = pd.concat(signals, axis=1).mean(axis=1)
        # print(avg)
        scores[rule.name] = avg.values

    if not scores:
        return pd.DataFrame(index=patients)
    return pd.DataFrame(scores, index=patients)

logfc_rule_score = compute_kg_rule_scores(logfc_score, rules, )
logfc_rule_score

,chain_SORL1_APP_CTL,chain_MME_APP_CTL,chain_CTSD_APP_AD,direct_BIN1_AD,direct_PICALM_AD,direct_CD33_AD,direct_ABCA7_AD,direct_GAB2_AD,chain_RTN4_APP_CTL,chain_CYLD_Wnt signaling pathwa_AD,...,chain_RAC1_5hops_CTL,chain_RAC1_5hops_AD,chain_RAC1_4hops_AD,chain_AKAP13_5hops_AD,chain_AKAP13_5hops_CTL,chain_AKAP13_4hops_AD,chain_VPS35_5hops_AD,chain_VPS35_5hops_CTL,chain_VPS35_4hops_AD,chain_GRN_3hops_AD
DCR00001_1,-0.008783,-0.002301,-0.044658,-0.073308,-0.046472,-0.126071,-0.031553,0.010670,-0.042811,-0.083182,...,0.005476,0.005476,0.005476,0.015545,0.015545,0.015545,-0.190746,-0.190746,-0.190746,-0.043529
DCR00004_1,-0.062480,-0.054695,-0.063441,-0.170913,-0.053998,-0.083542,-0.115499,-0.092522,0.006844,-0.001991,...,0.015144,0.015144,0.015144,-0.012680,-0.012680,-0.012680,-0.140889,-0.140889,-0.140889,-0.026627
DCR00009_1,-0.036563,-0.072370,-0.135399,-0.077809,0.035929,-0.204516,-0.073090,-0.020853,-0.019872,-0.078876,...,-0.025544,-0.025544,-0.025544,-0.055793,-0.055793,-0.055793,-0.090551,-0.090551,-0.090551,-0.054965
DCR00013_1,-0.004004,0.008637,-0.006385,-0.092820,0.007429,-0.181170,-0.063609,-0.029696,0.047643,-0.003935,...,-0.021242,-0.021242,-0.021242,0.000000,0.000000,0.000000,-0.155034,-0.155034,-0.155034,-0.094711
DCR00014_1,0.031838,0.012916,-0.070152,-0.040701,-0.153632,-0.162406,0.000720,-0.025512,0.010724,0.025586,...,-0.020439,-0.020439,-0.020439,-0.102395,-0.102395,-0.102395,-0.136067,-0.136067,-0.136067,-0.078308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TLSMCI607_1,0.107618,-0.019297,0.073416,0.193912,-0.004989,0.021269,0.022286,-0.038174,0.112625,0.328386,...,0.163070,0.163070,0.163070,0.215171,0.215171,0.215171,0.068189,0.068189,0.068189,0.028995
TLSMCI608_1,0.255401,0.182116,0.245009,0.118459,0.092948,0.123578,0.169381,0.215984,0.153507,0.284053,...,0.195088,0.195088,0.195088,0.356604,0.356604,0.356604,0.083825,0.083825,0.083825,0.216234
TLSMCI609_1,-0.047164,-0.041314,-0.245209,-0.065098,-0.038569,-0.228149,-0.189696,-0.070405,0.006420,-0.004011,...,-0.021816,-0.021816,-0.021816,-0.072847,-0.072847,-0.072847,-0.099530,-0.099530,-0.099530,-0.084082
TLSMCI611_1,-0.001777,-0.048393,-0.101306,-0.042896,-0.062395,-0.158562,-0.085843,-0.035917,-0.055252,-0.077405,...,-0.020725,-0.020725,-0.020725,-0.067209,-0.067209,-0.067209,-0.136070,-0.136070,-0.136070,-0.043511


## Build patient graph

In [110]:
from models import get_model
from rule_engine import extract_rules

In [112]:
# 1. extract rules from KG: kg_path, GE_genes
kg_path = "../data/KG/ad_network_with_reverse_edges.pkl"

exp_path = '../data/ANMerge/ANMerge_exp_cleaned.csv'
exp_df = pd.read_csv(exp_path, index_col=0)
expr_genes = set(exp_df.columns)

rules = extract_rules(kg_path = kg_path, 
                      expression_genes = expr_genes,
                      max_chain_length=4,
                      bp_mechanisms_path=None)
df_rules = summarise_rules(rules)
# get direct_rules
direct_rules = [r for r in rules if r.rule_type == 'direct_gene_ad']
direct_rules

  Direct gene→AD rules:    178
  Pathway activation rules: 8
  Causal chain rules:      553


[KGRule('chain_SORL1_APP_CTL', direct_gene_ad, → CTL),
 KGRule('chain_MME_APP_CTL', direct_gene_ad, → CTL),
 KGRule('chain_CTSD_APP_AD', direct_gene_ad, → AD),
 KGRule('direct_BIN1_AD', direct_gene_ad, → AD),
 KGRule('direct_PICALM_AD', direct_gene_ad, → AD),
 KGRule('direct_CD33_AD', direct_gene_ad, → AD),
 KGRule('direct_ABCA7_AD', direct_gene_ad, → AD),
 KGRule('direct_GAB2_AD', direct_gene_ad, → AD),
 KGRule('chain_RTN4_APP_CTL', direct_gene_ad, → CTL),
 KGRule('chain_CYLD_Wnt signaling pathwa_AD', direct_gene_ad, → AD),
 KGRule('chain_CSF1R_Microglia_AD', direct_gene_ad, → AD),
 KGRule('chain_PRNP_amyloid-beta_AD', direct_gene_ad, → AD),
 KGRule('chain_PRNP_BACE1_CTL', direct_gene_ad, → CTL),
 KGRule('chain_PRNP_APP_CTL', direct_gene_ad, → CTL),
 KGRule('chain_TLR4_inflammatory respons_CTL', direct_gene_ad, → CTL),
 KGRule('direct_IL1B_AD', direct_gene_ad, → AD),
 KGRule('chain_IL1B_amyloid-beta_AD', direct_gene_ad, → AD),
 KGRule('chain_IL1B_inflammatory respons_CTL', direct_gene

In [129]:
# 2. prepared log2fc_df to get features for building patient graph
exp = exp_df.T if exp_df.shape[0] < exp_df.shape[1] else exp_df
design = pd.read_csv("../data/ANMerge/design_2cls_3cl_target.csv", index_col=0)
ctl_ids = design[design['Target']=='Control'].index.to_list()
logfc_df = compute_log2fc(exp, ctl_ids)

# prepare features (kg_genes log2fc)
print('\nBuilding node features...')
kg_genes = sorted({g.upper() for r in rules for g in r.genes if g.upper() in logfc_df.index})
# logfc_kgrule_features
logfc_kgrule_features = logfc_df.loc[kg_genes].T.values
# logfc_allgene_features
logfc_features = logfc_df.T.values
# raw features
raw_features = exp.T.values

logfc_df


Building node features...


,DCR00001_1,DCR00004_1,DCR00009_1,DCR00013_1,DCR00014_1,DCR00015_1,DCR00018_1,DCR00020_1,DCR00021_1,DCR00025_1,...,TLSMCI602_1,TLSMCI603_1,TLSMCI604_1,TLSMCI605_1,TLSMCI606_1,TLSMCI607_1,TLSMCI608_1,TLSMCI609_1,TLSMCI611_1,TLSMCI612_1
GIMAP4,-0.038826,0.010775,-0.016693,-0.028975,-0.014859,-0.014026,0.002839,-0.042452,-0.446322,-0.474951,...,0.128418,0.127310,0.162567,0.181070,0.161564,0.181525,0.147258,0.006065,-0.033330,0.149625
C1QBP,-0.266329,-0.233575,-0.222817,-0.252745,-0.217130,-0.204301,-0.175709,-0.210039,0.348601,0.384706,...,-0.014236,-0.034085,-0.001396,-0.019894,0.018658,0.052203,-0.045519,-0.217794,-0.300428,-0.006148
OCIAD2,-0.269966,-0.193584,-0.151247,-0.178506,-0.102221,-0.135125,-0.139348,-0.084312,-0.092501,-0.152142,...,0.134316,0.167352,0.149753,0.185927,0.178041,0.281282,0.084772,-0.033706,-0.202318,0.096652
ERICH1,-0.047440,-0.004979,-0.039913,-0.090290,-0.071497,-0.068191,0.016536,-0.042061,-0.087378,-0.063139,...,0.209612,0.087394,0.203656,0.178588,0.242693,0.197776,0.193257,0.009480,-0.037132,0.271544
MRTFA,-0.287902,-0.344003,-0.315496,-0.157800,-0.253166,-0.310290,-0.375195,-0.308243,-0.001605,0.008548,...,0.096106,0.144493,0.097806,0.099971,0.073500,0.026924,0.115343,-0.442054,-0.213299,0.148500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SLC7A7,-0.030192,0.015155,0.020427,-0.032128,-0.098970,-0.047093,-0.025576,-0.017053,-0.386643,-0.423397,...,0.208758,0.173442,0.193019,0.208607,0.228475,0.138634,0.200028,-0.106328,-0.053148,0.175883
NAA10,-0.070484,-0.135928,-0.074766,-0.183371,-0.067519,-0.101231,-0.055748,-0.022156,0.387610,0.285040,...,0.085684,0.034564,0.093015,0.000959,0.017581,0.000807,0.010496,-0.075283,-0.047213,0.012061
ORAI1,-0.250984,-0.293897,-0.305138,-0.162962,-0.233402,-0.332424,-0.388120,-0.332308,0.015686,-0.051660,...,0.106590,0.095209,0.013234,0.112096,0.011277,0.071692,0.017792,-0.332566,-0.284617,0.029559
ZZeqdeed69UrlI7lNY,-0.156787,-0.322217,-0.025110,-0.345661,-0.263374,-0.223693,-0.353414,-0.287235,0.412736,0.384791,...,0.092717,0.159254,0.048066,0.128638,0.051123,0.059329,0.083046,-0.092700,-0.113970,0.136067


In [144]:
exp.min().min()

3.096392862

In [127]:
# 3. prepare node attributes: x, kg_scores, patient_id, y
# ── KG rule scores: symbolic node features ────────────────────────────────────
print('\nComputing KG rule scores per patient...')
kg_scores_df = compute_kg_rule_scores(logfc_df, rules)
kg_scores    = kg_scores_df.values   # (n_patients, n_rules)
print(f'  KG rule scores: {kg_scores.shape}')

# Also compute direct-rule-only scores (cleaner signal)
kg_scores_direct_df = compute_kg_rule_scores(logfc_df, direct_rules)
kg_scores_direct    = kg_scores_direct_df.values
print(f'  Direct-rule KG scores: {kg_scores_direct.shape}')
    


Computing KG rule scores per patient...
  KG rule scores: (691, 739)
  Direct-rule KG scores: (691, 178)


In [128]:
kg_scores_df

,chain_SORL1_APP_CTL,chain_MME_APP_CTL,chain_CTSD_APP_AD,direct_BIN1_AD,direct_PICALM_AD,direct_CD33_AD,direct_ABCA7_AD,direct_GAB2_AD,chain_RTN4_APP_CTL,chain_CYLD_Wnt signaling pathwa_AD,...,chain_RAC1_5hops_CTL,chain_RAC1_5hops_AD,chain_RAC1_4hops_AD,chain_AKAP13_5hops_AD,chain_AKAP13_5hops_CTL,chain_AKAP13_4hops_AD,chain_VPS35_5hops_AD,chain_VPS35_5hops_CTL,chain_VPS35_4hops_AD,chain_GRN_3hops_AD
DCR00001_1,-0.008783,-0.002301,-0.044658,-0.073308,-0.046472,-0.126071,-0.031553,0.010670,-0.042811,-0.083182,...,0.005476,0.005476,0.005476,0.015545,0.015545,0.015545,-0.190746,-0.190746,-0.190746,-0.043529
DCR00004_1,-0.062480,-0.054695,-0.063441,-0.170913,-0.053998,-0.083542,-0.115499,-0.092522,0.006844,-0.001991,...,0.015144,0.015144,0.015144,-0.012680,-0.012680,-0.012680,-0.140889,-0.140889,-0.140889,-0.026627
DCR00009_1,-0.036563,-0.072370,-0.135399,-0.077809,0.035929,-0.204516,-0.073090,-0.020853,-0.019872,-0.078876,...,-0.025544,-0.025544,-0.025544,-0.055793,-0.055793,-0.055793,-0.090551,-0.090551,-0.090551,-0.054965
DCR00013_1,-0.004004,0.008637,-0.006385,-0.092820,0.007429,-0.181170,-0.063609,-0.029696,0.047643,-0.003935,...,-0.021242,-0.021242,-0.021242,0.000000,0.000000,0.000000,-0.155034,-0.155034,-0.155034,-0.094711
DCR00014_1,0.031838,0.012916,-0.070152,-0.040701,-0.153632,-0.162406,0.000720,-0.025512,0.010724,0.025586,...,-0.020439,-0.020439,-0.020439,-0.102395,-0.102395,-0.102395,-0.136067,-0.136067,-0.136067,-0.078308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TLSMCI607_1,0.107618,-0.019297,0.073416,0.193912,-0.004989,0.021269,0.022286,-0.038174,0.112625,0.328386,...,0.163070,0.163070,0.163070,0.215171,0.215171,0.215171,0.068189,0.068189,0.068189,0.028995
TLSMCI608_1,0.255401,0.182116,0.245009,0.118459,0.092948,0.123578,0.169381,0.215984,0.153507,0.284053,...,0.195088,0.195088,0.195088,0.356604,0.356604,0.356604,0.083825,0.083825,0.083825,0.216234
TLSMCI609_1,-0.047164,-0.041314,-0.245209,-0.065098,-0.038569,-0.228149,-0.189696,-0.070405,0.006420,-0.004011,...,-0.021816,-0.021816,-0.021816,-0.072847,-0.072847,-0.072847,-0.099530,-0.099530,-0.099530,-0.084082
TLSMCI611_1,-0.001777,-0.048393,-0.101306,-0.042896,-0.062395,-0.158562,-0.085843,-0.035917,-0.055252,-0.077405,...,-0.020725,-0.020725,-0.020725,-0.067209,-0.067209,-0.067209,-0.136070,-0.136070,-0.136070,-0.043511


In [191]:
import os
import pickle
import json
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

def get_patient_features(
    kg_path: str, 
    exp_df: pd.DataFrame, 
    ctl_ids:list,
    patients: list,
    logfc: bool = True, 
    KGRule: bool = True
):
    """
    Extracts expression features and Knowledge Graph (KG) rule-based symbolic scores 
    for a given list of patients.

    Parameters:
        kg_path (str): Path to the knowledge graph file.
        exp_df (pd.DataFrame): Gene expression matrix (genes x samples or samples x genes).
        ctl_ids (list): Control group sample IDs.
        patients (list): List of patient IDs to extract features for.
        logfc (bool): Whether to use log-fold change transformated values as features. Defaults to True.
        KGRule (bool): Whether to restrict features to genes present in KG rules. Defaults to True.

    Returns:
        tuple: 
            - features (np.ndarray): Primary numerical feature matrix for patients.
            - kg_scores (np.ndarray): Full KG rule score matrix.
            - kg_scores_direct (np.ndarray): Direct-rule-only KG score matrix.
            - kg_genes (list): List of genes filtered through KG rules.
            - kg_scores_df (pd.DataFrame): DataFrame containing full rule scores.
            - kg_scores_direct_df (pd.DataFrame): DataFrame containing direct rule scores.
    """
    # 1. Extract rules from KG
    expr_genes = set(exp_df.columns)
    rules = extract_rules(
        kg_path=kg_path, 
        expression_genes=expr_genes,
        max_chain_length=4,
        bp_mechanisms_path=None
    )
    direct_rules = [r for r in rules if r.rule_type == 'direct_gene_ad']

    # 2. Prepare log2fc_df to get features for building patient graph
    exp = exp_df.T if exp_df.shape[0] < exp_df.shape[1] else exp_df
    logfc_df = compute_log2fc(exp, ctl_ids)

    # Prepare features
    print('\nBuilding node features...')
    if logfc and KGRule:
        kg_genes = sorted({g.upper() for r in rules for g in r.genes if g.upper() in logfc_df.index})
        features = logfc_df.loc[kg_genes][patients].T.values
    elif logfc and not KGRule:
        kg_genes = logfc_df.index.tolist()
        features = logfc_df.T.loc[patients].values
    else:
        kg_genes = exp.index.tolist()
        features = exp.T[patients].values

    # 3. KG rule scores: symbolic node features
    print('\nComputing KG rule scores per patient...')
    kg_scores_df = compute_kg_rule_scores(logfc_df[patients], rules)
    kg_scores = kg_scores_df.values  # (n_patients, n_rules)
    print(f' KG rule scores: {kg_scores.shape}')

    # Compute direct-rule-only scores
    kg_scores_direct_df = compute_kg_rule_scores(logfc_df[patients], direct_rules)
    kg_scores_direct = kg_scores_direct_df.values
    print(f' Direct-rule KG scores: {kg_scores_direct.shape}')
    
    return features, kg_scores, kg_scores_direct, kg_genes, kg_scores_df, kg_scores_direct_df


def build_patient_graph(
    kg_path: str,
    exp_df: pd.DataFrame,
    design_df: pd.DataFrame,
    k: int, 
    out_dir: str, 
    metric: str = 'cosine',
    logfc: bool = True,
    KGRule: bool = True
):
    """
    Constructs a patient similarity K-NN graph, attaches multi-modal node attributes 
    (expression features and symbolic KG rules), computes homophily, and saves the 
    graph object alongside metadata.

    Parameters:
        kg_path (str): Path to the knowledge graph file.
        exp_df (pd.DataFrame): Gene expression matrix.
        design_df (pd.DataFrame): Experimental design metadata frame.
        k (int): Number of nearest neighbors for graph construction.
        out_dir (str): Directory path to save the output graph and metadata.
        metric (str): Distance metric for K-NN (e.g., 'cosine', 'euclidean'). Defaults to 'cosine'.
        logfc (bool): Whether to use log-fold change. Defaults to True.
        KGRule (bool): Whether to use KG rule filtering. Defaults to True.

    Returns:
        nx.Graph: The constructed NetworkX patient graph.
    """
    patients = design_df.index.to_list()
    design_df['Target'] = design_df['Target'].map({'Disease':1,'Control':0})
    labels_mapping = design_df['Target'].to_dict()
    ctl_ids = design_df[design_df['Target']==0].index.to_list()
    
    # Fixed unpacking to match the updated return signature
    features, kg_scores, kg_scores_direct, kg_genes, kg_scores_df, kg_scores_direct_df = get_patient_features(
        kg_path, 
        exp_df, 
        ctl_ids,
        patients,
        logfc=logfc, 
        KGRule=KGRule
    )

    # K-NN clustering
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric=metric).fit(features)
    dists, inds = nbrs.kneighbors(features)

    G = nx.Graph()
    for i, pid in enumerate(patients):
        G.add_node(
            i, 
            x=features[i],
            patient_id=pid, 
            y=int(labels_mapping[pid]),
            kg_scores=kg_scores[i],
            kg_scores_direct=kg_scores_direct[i] 
        )

    for i in tqdm(range(len(patients)), desc='Adding edges'):
        for r in range(1, k + 1):
            j = inds[i, r]
            sim = 1.0 - dists[i, r]
            if i < j:
                G.add_edge(i, j, weight=float(sim))
    
    homophily = sum(G.nodes[u]['y'] == G.nodes[v]['y'] for u, v in G.edges()) / max(1, G.number_of_edges())
    print(f'  Nodes: {G.number_of_nodes()}  Edges: {G.number_of_edges()}')
    print(f'  Edge homophily: {homophily:.3f}')

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, 'patient_graph.gpickle')
    with open(out_path, 'wb') as f:
        pickle.dump(G, f)
    print(f'\nGraph saved to {out_path}')

    # Save metadata
    meta = {
        'nodes':G.number_of_nodes(),
        'edges':G.number_of_edges(),
        'edge_homophily':homophily,
        'kg_genes': kg_genes,
        'rule_names': kg_scores_df.columns.tolist(),
        'direct_rule_names': kg_scores_direct_df.columns.tolist(),
        'patients': patients,
        'labels': [int(labels_mapping[p]) for p in patients],
        'n_kg_genes': len(kg_genes),
        'n_rules': kg_scores.shape[1],
        'n_direct_rules': kg_scores_direct.shape[1],
    }
    meta_path = os.path.join(out_dir, 'graph_meta.json')
    # Write to a JSON file with nice indentation
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=4)
    
    return G,meta

In [182]:
# 1. extract rules from KG: kg_path, GE_genes
kg_path = "../data/KG/ad_network_with_reverse_edges.pkl"

exp_path = '../data/ANMerge/ANMerge_TX_preprocessed/AnmergeTX_cleaned_exp.csv'
exp_df = pd.read_csv(exp_path, index_col=0)
design_df = pd.read_csv("../data/ANMerge/design_2cls_3cl_target.csv", index_col=0)

patient_G, meta = build_patient_graph(
    kg_path=kg_path,
    exp_df=exp_df,
    design_df=design_df,
    k=5, 
    out_dir='./results', 
    metric= 'cosine',
    logfc= True,
    KGRule= True
)

  Direct gene→AD rules:    185
  Pathway activation rules: 9
  Causal chain rules:      547

Building node features...

Computing KG rule scores per patient...
 KG rule scores: (691, 741)
 Direct-rule KG scores: (691, 185)


Adding edges: 100%|██████████| 691/691 [00:00<00:00, 131143.17it/s]

  Nodes: 691  Edges: 1698
  Edge homophily: 0.610

Graph saved to ./results/patient_graph.gpickle


In [201]:
def compute_topological_features(G):
    """
    Compute topological features for all nodes in the graph.
    
    Args:
        G: NetworkX graph object
        
    Returns:
        dict: Dictionary containing topological features for each node
    """
    print("Computing topological features...")
    
    # Basic features
    degrees = np.array([G.degree(n) for n in G.nodes()])
    clustering = np.array([nx.clustering(G, n) for n in G.nodes()])
    
    # Centrality measures
    eigenvector_centrality = np.array(list(nx.eigenvector_centrality(G, weight='weight', max_iter=1000).values()))
    degree_centrality = np.array(list(nx.degree_centrality(G).values()))
    
    # Average edge weight
    avg_edge_weight = []
    for n in G.nodes():
        if G.degree(n) > 0:
            weights = [G.edges[n, m]['weight'] for m in G.neighbors(n)]
            avg_edge_weight.append(np.mean(weights))
        else:
            avg_edge_weight.append(0.0)
    avg_edge_weight = np.array(avg_edge_weight)
    
    return {
        'degrees': degrees,
        'clustering': clustering,
        'eigenvector_centrality': eigenvector_centrality,
        'degree_centrality': degree_centrality,
        'avg_edge_weight': avg_edge_weight
    }

## Convert graph to PyG Data

In [200]:
def build_pyg_data(G, use_direct_rules=False):
    """Convert NetworkX patient graph to PyTorch Geometric Data."""
    nodes = sorted(G.nodes())

    x           = np.array([G.nodes[n]['x'] for n in nodes])
    y           = np.array([G.nodes[n]['y'] for n in nodes])
    kg_scores   = np.array([G.nodes[n]['kg_scores_direct' if use_direct_rules
                                       else 'kg_scores'] for n in nodes])

    # Standardize node features
    scaler_x  = StandardScaler()
    scaler_kg = StandardScaler()
    x         = scaler_x.fit_transform(x)
    kg_scores = scaler_kg.fit_transform(kg_scores)

    # Edge index and weights
    edge_list  = []
    edge_attrs = []
    for u, v, d in G.edges(data=True):
        # include reverse edges
        edge_list.append([u, v]); edge_list.append([v, u])
        edge_attrs.append(d['weight']); edge_attrs.append(d['weight'])

    # Compute topological features from Graph
    topological_features = compute_topological_features(G)
    # Stack and standardize topological features
    feature_names = ['degrees', 'clustering',  
                    'eigenvector_centrality', 'degree_centrality', 'avg_edge_weight']
    
    topo_features = np.vstack([topological_features[name] for name in feature_names]).T
    scaler_topo = StandardScaler()
    topo_features = scaler_topo.fit_transform(topo_features)
    
    data = Data(
        x          = torch.tensor(x,         dtype=torch.float),
        y          = torch.tensor(y,         dtype=torch.long),
        kg_scores  = torch.tensor(kg_scores, dtype=torch.float),
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous(),
        edge_attr  = torch.tensor(edge_attrs,dtype=torch.float),
        topo_features=torch.tensor(topo_features, dtype=torch.float),
        topo_scaler_mean=torch.tensor(scaler_topo.mean_, dtype=torch.float),
        topo_scaler_scale=torch.tensor(scaler_topo.scale_, dtype=torch.float)
    )
    return data

In [202]:
data = build_pyg_data(patient_G, False)
data

Computing topological features...


Data(x=[691, 145], edge_index=[2, 3396], edge_attr=[3396], y=[691], kg_scores=[691, 741], topo_features=[691, 5], topo_scaler_mean=[5], topo_scaler_scale=[5])

In [198]:
data.x.size(0)
data.y

tensor([1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0,
        1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1,
        1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1,
        0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1,
        1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [206]:
def test(use_kg, use_topo):
    if use_kg and not use_topo:
        print(1, 'use_kg',use_kg, 'use_topo',use_topo)
    elif use_topo and not use_kg:
        print(2, 'use_kg',use_kg, 'use_topo',use_topo)
    elif use_topo and use_kg:
        print(3, 'use_kg',use_kg, 'use_topo',use_topo)
    else:
        print(4, 'use_kg',use_kg, 'use_topo',use_topo)
test(False, False)

4 use_kg False use_topo False
